In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [75]:
import fitz
from endee import Endee, Precision

from sentence_transformers import SentenceTransformer

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
GROQ_MODEL  = "llama-3.1-8b-instant"
INDEX_NAME  = "pdf_rag_index"   
CHUNK_SIZE  = 500               
TOP_K       = 4                 
TEMPERATURE = 0.2 

PDF_PATH = "Jayant_Yadav_Resume.pdf" 
GROQ_API_KEY = GROQ_API_KEY  ############ PUT YOUR GROQ API KEY HERE ######################

In [77]:
def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF file page by page."""
    doc = fitz.open(pdf_path)
    full_text = ""

    print(f"PDF has {len(doc)} page(s)")

    for page_num, page in enumerate(doc):
        page_text = page.get_text()
        full_text += page_text + "\n"
        print(f"Page {page_num + 1} → {len(page_text)} characters")

    doc.close()
    return full_text


print("Extracting text from PDF...\n")
raw_text = extract_pdf_text(PDF_PATH)

print(f"\nTotal characters extracted : {len(raw_text)}")
print(f"\nPREVIEW (first 500 chars)")
print(raw_text[:500])

Extracting text from PDF...

PDF has 1 page(s)
Page 1 → 3525 characters

Total characters extracted : 3526

PREVIEW (first 500 chars)
Jayant Yadav
 +91 8958348210 | # jy8316600@gmail.com | ï LinkedIn | § GitHub |  Twitter | Ð LeetCode |
Technical Profile
Computer Science student at Galgotias University with a heavy focus on Java-based problem solving and building end-to-end
ML systems. I’ve spent my time mastering data structures (300+ problems solved) and learning how to take models out of
notebooks and into production using Docker and Jenkins. Currently looking to apply my backend and MLOps skills to
large-scale engineerin


In [78]:
print(len(raw_text))
print(raw_text[:500])

3526
Jayant Yadav
 +91 8958348210 | # jy8316600@gmail.com | ï LinkedIn | § GitHub |  Twitter | Ð LeetCode |
Technical Profile
Computer Science student at Galgotias University with a heavy focus on Java-based problem solving and building end-to-end
ML systems. I’ve spent my time mastering data structures (300+ problems solved) and learning how to take models out of
notebooks and into production using Docker and Jenkins. Currently looking to apply my backend and MLOps skills to
large-scale engineerin


In [79]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list:
    """Split text into fixed-size word-boundary aligned chunks."""
    words = text.split()
    chunks, current_chunk, current_len = [], [], 0

    for word in words:
        current_chunk.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(" ".join(current_chunk))
            current_chunk, current_len = [], 0

    if current_chunk:                          # add  remaining words
        chunks.append(" ".join(current_chunk))

    return chunks


chunks = chunk_text(raw_text)

print(f"Created {len(chunks)} chunks")
print(f"\n--- Chunk 0 ---\n{chunks[0]}")
print(f"\n--- Chunk 1 ---\n{chunks[1]}")

Created 7 chunks

--- Chunk 0 ---
Jayant Yadav  +91 8958348210 | # jy8316600@gmail.com | ï LinkedIn | § GitHub |  Twitter | Ð LeetCode | Technical Profile Computer Science student at Galgotias University with a heavy focus on Java-based problem solving and building end-to-end ML systems. I’ve spent my time mastering data structures (300+ problems solved) and learning how to take models out of notebooks and into production using Docker and Jenkins. Currently looking to apply my backend and MLOps skills to large-scale engineering

--- Chunk 1 ---
projects. Education B.Tech CSE 2023 – 2027 Galgotias University, Greater Noida, Uttar Pradesh, India CGPA: 8.28 Experience Machine Learning Intern August 2025 – October 2025 Penstrike Inkworks, Kanpur • Analyzed book sales and reader behavior datasets using Python (Pandas, NumPy) to uncover purchasing trends and genre preferences. data preprocessing, feature engineering, and exploratory data analysis (EDA) with Pandas, Matplotlib, and Seaborn 

In [80]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Quick sanity check
test_vec = embedder.encode(["Hello world"])[0]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12035.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [81]:
print(f"   Vector dimensions : {len(test_vec)}")

   Vector dimensions : 384


In [ ]:

endee_client = Endee()  
print(" Connected to Endee at localhost:8080")


try:
    endee_client.create_index(
        name=INDEX_NAME,
        dimension=384,         
        space_type="cosine",   
        precision=Precision.INT8
    )
    print(f"Index '{INDEX_NAME}' created!")

except Exception:
    print(f"Index '{INDEX_NAME}' already exists — reusing it.")


index = endee_client.get_index(name=INDEX_NAME)
print(f"Index ready: '{INDEX_NAME}'")

🔌 Connected to Endee at localhost:8080
Index 'pdf_rag_index' already exists — reusing it.
Index ready: 'pdf_rag_index'


In [ ]:
print(f"Embedding {len(chunks)} chunks...\n")

# Embed all chunks in one batched call
vectors = embedder.encode(chunks, show_progress_bar=True)

print(f"\nUpserting into Endee...")

vector_items = [
    {
        "id": f"chunk_{i}",
        "vector": vec.tolist(),
        "meta": {"text": chunk, "chunk_id": i}
    }
    for i, (chunk, vec) in enumerate(zip(chunks, vectors))
]

index.upsert(vector_items)

print(f"\n{len(chunks)} chunks stored in Endee!")
print("   Your PDF is now a searchable vector index.")

Embedding 7 chunks...



Batches: 100%|██████████| 1/1 [00:00<00:00,  5.45it/s]


Upserting into Endee...

7 chunks stored in Endee!
   Your PDF is now a searchable vector index. 🎉


In [84]:
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=TEMPERATURE
)

In [85]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant that answers questions strictly based on the provided context.

Rules:
- Answer ONLY using the context below.
- If the answer is not in the context, say: 'I could not find relevant information in the document.'
- Be concise and clear.

Context:
{context}"""
    ),
    (
        "human",
        "{question}"
    )
])
rag_chain = prompt_template | llm | StrOutputParser()


In [86]:
def rag_query(question: str) -> str:
    """
    Full RAG pipeline using Endee + LangChain + Groq.

    Steps:
      1. Embed the question locally
      2. Search Endee for top-k similar chunks
      3. Inject chunks into LangChain prompt template
      4. Run chain → Groq LLM → plain string answer
    """
    print(f"\Question: {question}")
    print("-" * 55)

    # 1. Embed the question 
    print("[1/3] Embedding question with sentence-transformers...")
    query_vector = embedder.encode([question])[0].tolist()

    # 2. Search Endee 
    print(f"[2/3] Searching Endee (top {TOP_K} chunks)...")
    results = index.query(vector=query_vector, top_k=TOP_K)

    
    context = "\n\n".join([
        f"[Chunk {r['meta']['chunk_id']}]\n{r['meta']['text']}"
        for r in results
    ])

    print(f"\n--- Retrieved Context (preview) ---")
    print(context[:600])
    print("..." if len(context) > 600 else "")

    #3. Run LangChain chain
    print(f"\n   [3/3] Running LangChain → Groq ({GROQ_MODEL})...")
    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return answer

In [87]:
my_question = "What is the main topic of this document?"  

answer = rag_query(my_question)

print(f"\n{'=' * 60}")
print("FINAL ANSWER:")
print(f"{'=' * 60}")
print(answer)

\Question: What is the main topic of this document?
-------------------------------------------------------
[1/3] Embedding question with sentence-transformers...
[2/3] Searching Endee (top 4 chunks)...

--- Retrieved Context (preview) ---
[Chunk 1]
projects. Education B.Tech CSE 2023 – 2027 Galgotias University, Greater Noida, Uttar Pradesh, India CGPA: 8.28 Experience Machine Learning Intern August 2025 – October 2025 Penstrike Inkworks, Kanpur • Analyzed book sales and reader behavior datasets using Python (Pandas, NumPy) to uncover purchasing trends and genre preferences. data preprocessing, feature engineering, and exploratory data analysis (EDA) with Pandas, Matplotlib, and Seaborn to prepare data for modeling. • Developed a book recommendation

[Chunk 6]
Pandas, NumPy, Matplotlib, Seaborn, Power BI, Flask, FastAPI, Git, G
...

   [3/3] Running LangChain → Groq (llama-3.1-8b-instant)...

FINAL ANSWER:
The main topic of this document appears to be a technical resume or profile of 